# 03 — Python Data Processing & Analytics

**Module:** Databases and Analytics
**Case study:** NorthStar Urban Mobility & Logistics
**Learning outcomes covered:** LO1 (analysis), LO4 (analytics with Python).

This notebook performs end-to-end data processing in **Python with pandas, NumPy, matplotlib, and seaborn**.
We replicate the SQL/R findings, add deeper analytical work (cohort analysis, anomaly detection,
correlation analysis), and prepare the cleaned dataframes for ingestion into MongoDB.


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings; warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
PALETTE = ["#1f4e79", "#c0504d", "#9bbb59", "#f79646", "#8064a2", "#4bacc6"]
sns.set_palette(PALETTE)

DATA = "data"   # adjust to "/content/data" in Colab after upload


## 2. Load all relational tables

In [ ]:
zones    = pd.read_csv(f"{DATA}/zones.csv")
hubs     = pd.read_csv(f"{DATA}/hubs.csv")
vehicles = pd.read_csv(f"{DATA}/vehicles.csv")
drivers  = pd.read_csv(f"{DATA}/drivers.csv")
customers= pd.read_csv(f"{DATA}/customers.csv")
journeys = pd.read_csv(f"{DATA}/journeys.csv", parse_dates=["date"])
charging = pd.read_csv(f"{DATA}/charging_sessions.csv", parse_dates=["date"])
maintenance = pd.read_csv(f"{DATA}/maintenance.csv", parse_dates=["date_logged"])
complaints  = pd.read_csv(f"{DATA}/complaints.csv", parse_dates=["submitted_date"])

for name, df in [("zones",zones),("hubs",hubs),("vehicles",vehicles),
                 ("drivers",drivers),("customers",customers),("journeys",journeys),
                 ("charging",charging),("maintenance",maintenance),("complaints",complaints)]:
    print(f"  {name:12s} {df.shape[0]:>7,} rows  ×  {df.shape[1]} cols")


## 3. Data quality checks

In [ ]:
# Missing values across all tables
mv = {name: df.isna().sum().sum() for name, df in
      [("journeys",journeys),("charging",charging),("maintenance",maintenance),
       ("complaints",complaints),("vehicles",vehicles),("drivers",drivers),
       ("customers",customers),("zones",zones),("hubs",hubs)]}
print("Missing values per table:")
for k,v in mv.items(): print(f"  {k:12s} {v}")

# Referential integrity check: every journey's zone exists in zones
orphan = ~journeys.zone_id.isin(zones.zone_id)
print(f"\nOrphan journey zones: {orphan.sum()}")


## 4. Feature engineering

In [ ]:
# Add derived columns
journeys["profit"]       = journeys["revenue_gbp"] - journeys["cost_gbp"]
journeys["failed"]       = (journeys["status"] == "Failed").astype(int)
journeys["late_or_fail"] = (journeys["status"] != "Completed").astype(int)
journeys["weekday"]      = journeys["date"].dt.day_name()
journeys["is_weekend"]   = journeys["date"].dt.weekday >= 5

# Enrich with city
journeys = journeys.merge(zones[["zone_id","city","is_problem_zone"]], on="zone_id")

journeys.head()


## 5. Group-by analysis — replicate / extend the SQL findings

In [ ]:
# Performance by city × service
perf = journeys.groupby(["city","service_type"]).agg(
    journeys=("journey_id","count"),
    fail_pct=("failed", lambda s: round(s.mean()*100, 2)),
    avg_delay=("delay_minutes", "mean"),
    total_profit=("profit","sum")
).round(2).reset_index().sort_values("fail_pct", ascending=False)
perf.head(15)


In [ ]:
# Top 10 worst-performing zones by failure rate (min 100 journeys)
zone_perf = journeys.groupby(["zone_id","city"]).agg(
    journeys=("journey_id","count"),
    fail_pct=("failed", lambda s: round(s.mean()*100, 2)),
    avg_delay=("delay_minutes","mean"),
    profit=("profit","sum")
).reset_index()
worst = zone_perf[zone_perf.journeys >= 100].sort_values("fail_pct", ascending=False).head(10)
worst


## 6. Visualisation: integrated dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (a) Failure rate by city
city_fail = journeys.groupby("city")["failed"].mean().mul(100).sort_values(ascending=False)
axes[0,0].bar(city_fail.index, city_fail.values, color="#c0504d")
axes[0,0].set_title("Failure rate by city (%)")
axes[0,0].set_ylabel("%")
for i,v in enumerate(city_fail.values):
    axes[0,0].text(i, v+0.1, f"{v:.1f}", ha="center", fontsize=9)

# (b) Delay distribution by service type
sns.boxplot(data=journeys, x="service_type", y="delay_minutes", ax=axes[0,1])
axes[0,1].set_title("Delay distribution by service type")
axes[0,1].set_xlabel("")
axes[0,1].tick_params(axis="x", rotation=15)

# (c) Profit by service type
prof_by_svc = journeys.groupby("service_type")["profit"].sum().sort_values()
colors = ["#c62828" if v < 0 else "#4caf50" for v in prof_by_svc.values]
axes[1,0].barh(prof_by_svc.index, prof_by_svc.values, color=colors)
axes[1,0].set_title("Total profit by service type (90 days, GBP)")
axes[1,0].axvline(0, color="black", linewidth=0.8)

# (d) Override impact on failure
ovr = journeys.groupby("manual_override")["failed"].mean().mul(100)
axes[1,1].bar(["No override","Manual override"], ovr.values, color=["#4caf50","#c62828"])
axes[1,1].set_title("Failure rate (%) — override effect")
for i,v in enumerate(ovr.values):
    axes[1,1].text(i, v+0.2, f"{v:.1f}%", ha="center")

plt.suptitle("NorthStar — operational performance dashboard", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Anomaly detection — charging hub ghost sessions

In [ ]:
ghost = charging.groupby("hub_id").agg(
    sessions=("session_id","count"),
    total_kwh=("kwh_delivered","sum"),
    ghost_rate=("asset_monitor_flag", lambda s: round((s==0).mean()*100, 2))
).reset_index()

# Z-score over ghost rate; flag hubs > 2 sigma
mu, sd = ghost.ghost_rate.mean(), ghost.ghost_rate.std()
ghost["z"] = (ghost.ghost_rate - mu) / sd
ghost["anomaly"] = ghost.z > 2

print(f"Mean ghost rate: {mu:.2f}%   std: {sd:.2f}")
print(f"Anomalous hubs (z>2): {ghost.anomaly.sum()}")
ghost[ghost.anomaly].sort_values("ghost_rate", ascending=False)


## 8. Cohort: complaint follow-through by loyalty tier

In [ ]:
cust_complaints = complaints.merge(customers[["customer_id","loyalty_tier","customer_type"]],
                                   on="customer_id", how="left")
cohort = cust_complaints.groupby("loyalty_tier").agg(
    complaints=("complaint_id","count"),
    resolved_pct=("resolved", lambda s: round(s.mean()*100, 2))
).reset_index()
cohort


## 9. Correlation analysis

In [ ]:
num = journeys[["scheduled_minutes","actual_minutes","delay_minutes",
                "manual_override","is_problem_zone","revenue_gbp","cost_gbp",
                "profit","failed"]]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix — journey numeric features")
plt.tight_layout(); plt.show()


`delay_minutes` correlates with `manual_override` (~0.18) and `is_problem_zone` (~0.20). Failure correlates most with `is_problem_zone`. The signals match the regression in notebook 02.

## 10. Export cleaned outputs (used by MongoDB notebook)

In [ ]:
journeys.to_csv("data/journeys_enriched.csv", index=False)
zone_perf.to_csv("data/zone_performance.csv", index=False)
perf.to_csv("data/city_service_performance.csv", index=False)
ghost.to_csv("data/charging_hub_anomaly.csv", index=False)
print("Exported enriched files for downstream use.")


## 11. Summary

* The Python pipeline confirms and **quantifies** the SQL/R findings: failure rate is concentrated in 2 cities and a handful of zones.
* Two service lines (last-mile delivery in specific cities) are unprofitable once full cost is attributed.
* ~12% of charging sessions are 'ghost'; a small set of hubs are clear outliers and should be audited.
* These outputs feed the MongoDB design (notebook 04), where customer cases will carry the journey, zone, and override context that today is fragmented across 4+ systems.
